# One sequence in, K candidates out

Upload the antibody and the antigen, get a ranked, gated, structure-vetoed set of
designs and a submission file. This is `examples/maturation.py` and
`examples/track_a_fold.py` joined up, starting from sequences instead of a PDB
accession, with the stages of `docs/filtering_strategy.md` in order.

**This notebook needs a GPU and the model environments.** There is no fixture
mode: every number it prints comes from a model that really ran. The first cell
after the parameters runs `design_pipeline.preflight` for exactly the adapters
named below and stops if one of them does not resolve, so a missing environment
fails in ten seconds instead of in the middle of a 4,598-row scan.

What it does, in order:

| | Step | Call |
|---|---|---|
| 0 | format checks on the uploaded sequences | `track_a_fold.clean`, tag motifs, length |
| 1 | fold the parent complex over several seeds | `get_structure(antibody_seq=, antigen_seq=)` |
| 2 | read the chain roles back off the prediction | `structures.chain_ids` / `chain_sequence` |
| 3 | every single mutation of the chain being matured | `enumerate_variants` |
| 4 | fan out over the scorers, one column each | `score_variants` |
| 5 | drop the confident developability failures | `gate` |
| 6 | combine survivors into pairs far enough apart | `select_pairs` + `score_variants` |
| 7 | structure veto: refold the designs, several seeds | `examples/track_a_fold.py` |
| 8 | pick K, diversity-constrained, write the submission | this notebook |

Nothing is averaged into a combined score anywhere, and no pair's score is
guessed from its singles'. Which column to trust is a decision you make in the
`RANK_BY` parameter, and `design_pipeline.harness` is where that decision should
come from.

In [ ]:
# --- parameters ---------------------------------------------------------------
# The only cell meant to be edited. Paths are relative to the repository root.

# The upload. A FASTA file or a bare sequence string, either way.
# Give the antibody as TWO records - VH then VL - where you have them: the fold
# then carries separate heavy and light chains, which is what AntiFold's chain
# flags expect. A single scFv record works, and the input cell says what it costs.
ANTIBODY = "data/inputs/antibody.fasta"
ANTIGEN = "data/inputs/antigen.fasta"

# How much of the antigen to fold. A whole ectodomain is several hundred residues
# and fold time grows with length. Both are 1-based and numbered as in the
# sequence you pass; give one or neither.
ANTIGEN_REGION = None            # e.g. "480-607"
EPITOPE = None                   # e.g. "557,558,560-565", kept with PAD either side
PAD = 20

# Step 1, the parent fold.
PREDICTOR = "boltz"              # or "chai", "protenix", "colabfold", "alphafold3"
NUM_SEEDS = 5
DECOY_ANTIGEN = None             # an unrelated protein: the control that says whether
                                 # this ipTM can tell your antigen from any protein

# Step 3, the scan. Which uploaded antibody record to mature: 0 is the first.
SCAN_RECORD = 0
POSITIONS = None                 # None scans the whole chain, minus the exclusions
EXCLUDE_LINKER = True            # an scFv's (G4S)n linker is not a binding position
EXCLUDE_CYSTEINE = True          # a lost disulfide is not an affinity experiment

# Step 4, the scorers. One `score_<model>` column each, joined on variant_id,
# never averaged. DEFAULT_SCORERS is ("antifold", "ablang2", "consensus").
SCORERS = ["antifold", "ablang2", "consensus"]
RANK_BY = None                   # None takes the first score column that exists
MAX_PER_VARIANT = 200            # cap for a per-variant scorer on the singles table
# AntiFold is scored through this repository's own `antifold_score` module, which
# needs AntiFold's source folder - the one holding models/model.pt. None takes it
# from $ANTIFOLD_SRC, which is the normal way to set it.
ANTIFOLD_SRC = None

# Step 5, the gate.
GATE_CHECKS = ["liabilities"]

# Step 6, pairs. antifold_seq rather than antifold: the matrix lookup is exact for
# one mutation and has no answer for two, and nothing here sums two singles.
MAX_PAIRS = 20
MIN_DISTANCE = 10.0              # angstroms, CA to CA, read off the structure
PAIR_SCORERS = ["antifold_seq"]

# Step 7, the structure veto.
TOP_SINGLES = 10                 # best singles carried into the veto, beside the pairs
VETO_SEEDS = 5                   # the veto needs several: it compares against seed noise
VETO_SD_MULTIPLE = 2.0           # drop a design below parent mean ipTM - n x parent SD

# Step 8, the pick.
K = 10                           # designs to submit
MAX_SHARED_POSITIONS = 1         # positions a design may share with one already kept

# Two chains are written back out as one scFv with this linker, because the veto
# fold and the submission are both one antibody chain. "" keeps them separate and
# skips the veto.
SUBMIT_LINKER = "GGGGS" * 3

LENGTH_LIMIT = None              # the announced submission length limit, if there is one
RUN = "runs/end_to_end"

## Setup and preflight

`preflight` prints one row per adapter this run needs and what was probed for it.
It imports nothing for real and downloads nothing. A row that is not `yes` stops
the notebook here: the point of running it first is that a missing environment
costs ten seconds rather than an hour.

`(route unchecked)` beside a `NO` means the install route exists but nobody has
verified it on this machine - still a stop, because an unresolved adapter cannot
score anything.

In [ ]:
import importlib.util
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

try:  # Jupyter supplies it; this keeps the notebook runnable headless.
    display
except NameError:
    def display(obj):
        print(obj)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "pyproject.toml").is_file():
            return folder
    raise RuntimeError(f"no pyproject.toml above {here}: run this from the repository")


REPO = repo_root()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from design_pipeline import (  # noqa: E402
    confidence as confidence_module,
    enumerate_variants,
    gate,
    get_structure,
    preflight,
    score_variants,
    select_pairs,
    structures,
)
from design_pipeline.registry import get_adapter  # noqa: E402
from design_pipeline.variants import (  # noqa: E402
    mutant_sequence,
    parse_id,
    read_table,
)

# examples/track_a_fold.py is a script, not a package. Its sequence reading,
# antigen trimming and tag warnings are the tested ones, so they are loaded by
# path and reused rather than written a second time here.
_spec = importlib.util.spec_from_file_location(
    "track_a_fold", REPO / "examples" / "track_a_fold.py"
)
track_a = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(track_a)
TRACK_A_FOLD = REPO / "examples" / "track_a_fold.py"

RUN = Path(RUN)
RUN.mkdir(parents=True, exist_ok=True)
print(f"repository {REPO}")
print(f"run        {RUN}\n")

needed = [PREDICTOR, *SCORERS, *PAIR_SCORERS, *GATE_CHECKS]
seen: list[str] = []
for name in needed:
    if name not in seen:
        seen.append(name)
table = preflight.rows(seen)
print(preflight.render(table))
for note in preflight.notes(seen):
    print(note)

unresolved = [row[0] for row in table if not str(row[3]).startswith("yes")]
if unresolved:
    raise RuntimeError(
        "these adapters do not resolve here: "
        + ", ".join(unresolved)
        + ". INSTALL.md has the routes; the `how` column above says what was probed."
    )
print("\nevery adapter this run needs resolves.")

## 0. The upload, and the format checks

Stage 0 of `docs/filtering_strategy.md`. `track_a_fold.clean` refuses anything
outside the 20 standard amino acids, naming the residue and its position while
the sequence is still on screen - Boltz would otherwise fail deep inside the run.
The tag motifs are a warning, not a stop: Adaptyv appends its own 64-residue
Twin-Strep tag, so a sequence that already carries one would be tagged twice.

In [ ]:
antibody_records = track_a.read_sequences(str(ANTIBODY))
antigen_records = track_a.read_sequences(str(ANTIGEN))

if len(antigen_records) != 1:
    raise ValueError(f"ANTIGEN holds {len(antigen_records)} records; give exactly one")
if not 1 <= len(antibody_records) <= 2:
    raise ValueError(
        f"ANTIBODY holds {len(antibody_records)} records; give one scFv, or VH then VL"
    )
if not 0 <= SCAN_RECORD < len(antibody_records):
    raise ValueError(f"SCAN_RECORD={SCAN_RECORD} but there are {len(antibody_records)} records")

antigen_name, antigen_full = antigen_records[0]
region = track_a.parse_region(ANTIGEN_REGION) if ANTIGEN_REGION else None
epitope = track_a.parse_residues(EPITOPE) if EPITOPE else None
antigen, antigen_start, antigen_end = track_a.trim_antigen(
    antigen_full, region, epitope, PAD
)

TAGS = {"Twin-Strep": track_a.STREP_MOTIF, "His6": "HHHHHH", "FLAG": "DYKDDDDK"}
print(f"{'record':<24}{'len':>6}  first 30 residues")
print("-" * 62)
for name, seq in antibody_records:
    print(f"{name:<24}{len(seq):>6}  {seq[:30]}")
for name, seq in antibody_records:
    for tag, motif in TAGS.items():
        if motif in seq:
            print(f"warning: {name} contains the {tag} motif {motif}")
    if LENGTH_LIMIT is not None and len(seq) > LENGTH_LIMIT:
        print(f"warning: {name} is {len(seq)} residues, over the {LENGTH_LIMIT} limit")

print(
    f"\nantigen {antigen_name}: {len(antigen_full)} residues, "
    f"folding {antigen_start}-{antigen_end} ({len(antigen)} kept)"
)
for message in track_a.warnings_for(antibody_records, len(antigen)):
    print(f"warning: {message}")

if len(antibody_records) == 1:
    print(
        "\nOne antibody record, so the fold has one antibody chain. AntiFold's\n"
        "--heavy_chain / --light_chain flags describe a paired Fv, and this run can\n"
        "only give it the one chain - it may refuse, and where it does not, it is\n"
        "scoring an unpaired model. Giving VH and VL as two records is the better\n"
        "input; SUBMIT_LINKER joins them again for the veto fold and the submission."
    )

## 1. Fold the parent

`get_structure` with sequences instead of `pdb_id`. The chains are written to one
FASTA joined with `:`, which is the input every folding adapter here reads, and
every seed becomes a path on the returned `Complex` - `complex_.path` is the
tool's own top pick and `complex_.paths` is the ensemble behind it.

`confidence` is the thing to read before trusting anything downstream.
`dominant_cluster_fraction` is how many seeds agree on one interface, which is
not self-assessment and is therefore worth more than ipTM. With
`DECOY_ANTIGEN` set, `decoy_gap` says whether this ipTM can tell your antigen
from an unrelated protein; a gap near zero means the number cannot.

In [ ]:
antibody_seq = [seq for _, seq in antibody_records]
complex_ = get_structure(
    RUN / "parent",
    model_name=PREDICTOR,
    antibody_seq=antibody_seq,
    antigen_seq=antigen,
    num_seeds=NUM_SEEDS,
    decoy_antigen_seq=DECOY_ANTIGEN,
)

print(f"representative: {complex_.path}")
print(f"ensemble:       {len(complex_)} seed(s)")
if complex_.confidence:
    display(pd.Series(dict(complex_.confidence), name="confidence").to_frame())
    for message in confidence_module.warn(complex_.confidence):
        print(f"warning: {message}")

## 2. Which chain is which

The predictor labels the chains itself - `chain_ids=` is recorded on the
`Complex` but no folding adapter applies it - so the roles are read back off the
file by matching the sequences that went in. This is the cell that prevents the
expensive mistake: a wrong chain letter here scores 4,598 variants at positions
that belong to another molecule, and nothing downstream would notice.

In [ ]:
labels = structures.chain_ids(complex_.path)
sequences = {chain: structures.chain_sequence(complex_.path, chain) for chain in labels}


def identity(a: str, b: str) -> float:
    if len(a) != len(b) or not a:
        return 0.0
    return sum(x == y for x, y in zip(a, b)) / len(a)


def role_of(query: str, label: str) -> str:
    exact = [chain for chain, seq in sequences.items() if seq == query]
    if len(exact) == 1:
        return exact[0]
    scored = sorted(
        ((identity(seq, query), chain) for chain, seq in sequences.items()), reverse=True
    )
    best, chain = scored[0]
    runner_up = scored[1][0] if len(scored) > 1 else 0.0
    if best < 0.9 or best - runner_up < 0.1:
        raise RuntimeError(
            f"cannot tell which predicted chain is {label}: best identity {best:.2f} "
            f"on chain {chain}, next {runner_up:.2f}. The chains are printed above; "
            f"set the roles by hand in this cell."
        )
    print(f"note: {label} matched chain {chain} at identity {best:.2f}, not exactly")
    return chain


print(f"{'chain':<7}{'len':>6}  role")
print("-" * 34)
ab_chains = [
    role_of(seq, name) for name, seq in antibody_records
]
antigen_chain = role_of(antigen, antigen_name)
roles = {chain: f"antibody ({antibody_records[i][0]})" for i, chain in enumerate(ab_chains)}
roles[antigen_chain] = f"antigen ({antigen_name})"
for chain in labels:
    print(f"{chain:<7}{len(sequences[chain]):>6}  {roles.get(chain, 'UNASSIGNED')}")

if len(set(ab_chains) | {antigen_chain}) != len(ab_chains) + 1:
    raise RuntimeError(f"two roles matched one chain: antibody {ab_chains}, antigen {antigen_chain}")

scan_chain = ab_chains[SCAN_RECORD]
heavy_chain = ab_chains[0]
light_chain = ab_chains[1] if len(ab_chains) > 1 else None
print(f"\nmaturing chain {scan_chain} ({antibody_records[SCAN_RECORD][0]})")

# Scorer arguments. These go to every model named in one score_variants call, so
# the scoring cell makes one call per model and chains the tables together.
SCORER_KWARGS: dict[str, dict] = {
    # `antifold` is not AntiFold's own command line: that one takes no candidates
    # CSV and writes a per-residue matrix, so the adapter runs this repository's
    # antifold_score module on AntiFold's table route instead. The table route
    # finds the chains itself, which is why there are no chain flags here. What
    # it does need is AntiFold's source folder, for models/model.pt.
    "antifold": {"antifold_src": ANTIFOLD_SRC},
    # `antifold_seq` is AntiFold's own score_sequence_in_complex, one forward pass
    # per mutated chain, and that one does need the chain ids.
    "antifold_seq": {"heavy": heavy_chain, "light": light_chain or heavy_chain},
    "ablang2": {"antibody_chains": ab_chains},
    "antiberty": {"antibody_chains": ab_chains},
}

## 3. Every single mutation of that chain

The full scan is 19 mutations per position - 4,598 rows for a 242-residue chain.
Two exclusions, both because the position is not an affinity experiment: the
`(G4S)n` linker of an scFv, and the cysteines, where a mutation removes a
disulfide and changes the fold rather than the interface.

In [ ]:
sequence = sequences[scan_chain]
wanted = list(POSITIONS) if POSITIONS is not None else list(range(1, len(sequence) + 1))

excluded: dict[str, set[int]] = {}
if EXCLUDE_LINKER:
    excluded["linker"] = {
        i
        for match in re.finditer(r"(?:G{3,4}S){2,}", sequence)
        for i in range(match.start() + 1, match.end() + 1)
    }
if EXCLUDE_CYSTEINE:
    excluded["cysteine"] = {i for i, aa in enumerate(sequence, start=1) if aa == "C"}

drop = set().union(*excluded.values()) if excluded else set()
positions = [p for p in wanted if p not in drop]
for label, found in excluded.items():
    inside = sorted(found & set(wanted))
    shown = ", ".join(str(p) for p in inside[:12]) + (" ..." if len(inside) > 12 else "")
    print(f"excluded {len(inside):>3} {label} position(s): {shown or '(none)'}")
if not positions:
    raise ValueError("every position was excluded; relax POSITIONS or the exclusions")

singles = enumerate_variants(
    complex_, chain=scan_chain, positions=positions, output_dir=RUN / "variants"
)
print(f"\n{len(positions)} position(s) scanned -> {singles.candidates} candidates")
print(f"table: {singles.path}")
display(singles.frame.head())

## 4. Score

One call per scorer, chained, because `score_variants` passes `**kwargs` to every
model it is given and these scorers take different ones - a flag one of them
rejects would fail the whole call. Each call joins its column onto the table the
previous one returned, which is the same join the fan-out does inside one call.

Caching is per scorer on (structure, candidate set, model, kwargs), so adding a
scorer to `SCORERS` and re-running this cell does not re-run the ones that
already finished.

In [ ]:
scored = singles
for model in SCORERS:
    adapter = get_adapter(model)
    per_variant = bool(getattr(adapter, "per_variant", False))
    cap = MAX_PER_VARIANT if per_variant else None
    kwargs = {k: v for k, v in SCORER_KWARGS.get(model, {}).items() if v is not None}
    detail = f" (per variant, capped at {cap})" if per_variant else ""
    print(f"running {model}{detail} ...")
    scored = score_variants(
        complex_,
        scored,
        output_dir=RUN / "scores",
        models=[model],
        max_variants=cap,
        **kwargs,
    )
    print(f"  columns now: {', '.join(scored.score_columns)}")

frame = scored.frame
coverage = {c: int(frame[c].notna().sum()) for c in scored.score_columns}
print(f"\n{scored.candidates} candidates; scored per column: {coverage}")

## 5. Gate

Each check answers a boolean and a reason. A failure is dropped, never traded off
against binding score. The annotated table beside the survivors keeps every
candidate and every reason, so a gate that dropped too much can be re-tuned from
the file instead of by re-running the tools.

In [ ]:
survivors = gate(scored, checks=GATE_CHECKS, output_dir=RUN / "gated")
print(f"survivors: {survivors.candidates} of {scored.candidates}")

annotated = read_table(Path(survivors.config["annotated"]))
for check in GATE_CHECKS:
    column, reason = f"pass_{check}", f"reason_{check}"
    if column not in annotated:
        continue
    failed = annotated[~annotated[column].astype(bool)]
    print(f"\n{check}: {len(failed)} dropped")
    if len(failed):
        display(failed[reason].value_counts().head(10).to_frame("candidates"))

## 6. Rank, and look at whether the scorers agree

The ranking column is a decision, not a default: there is no combined score in
this package and no weights. `design_pipeline.harness` is where the choice should
come from - Spearman and precision@10 per scorer on systems with published
affinities.

The correlation below is not that. It says whether these scorers agree with each
other on this one antibody, which is worth knowing before you trust a column
nobody has calibrated: three columns that agree are one opinion, and three that
disagree tell you the ranking is a guess.

In [ ]:
frame = survivors.frame
columns = list(survivors.score_columns)
rank_by = RANK_BY or (columns[0] if columns else None)
if rank_by is None:
    raise RuntimeError("no score columns on the table; nothing to rank on")
if rank_by not in frame.columns:
    raise KeyError(f"no column {rank_by!r}; present: {', '.join(columns)}")
print(f"ranking on {rank_by}")

ranked = frame.sort_values(rank_by, ascending=False, kind="stable")
shown = ["variant_id", "chain", "position", "imgt", "wt", "mut", *columns]
display(ranked[[c for c in shown if c in ranked]].head(TOP_SINGLES))

if len(columns) > 1:
    print("\nSpearman between the score columns, on these survivors:")
    display(frame[columns].corr(method="spearman").round(2))

## 7. Pairs

Candidates are taken in the table's row order and combined in triangular order,
so `MAX_PAIRS` spreads across the top of the ranking instead of every pair
containing the single best mutation. `MIN_DISTANCE` is CA to CA off the
structure, which is why the structure travels with the table.

`select_pairs` drops the score columns on purpose. A pair's score is not a
function of its singles', so the pairs are scored again - and with
`antifold_seq`, which conditions on the whole mutant chain in one pass, rather
than with the matrix lookup that is exact only for a single mutation.

In [ ]:
pairs = select_pairs(
    survivors,
    max_pairs=MAX_PAIRS,
    min_distance=MIN_DISTANCE,
    output_dir=RUN / "pairs",
    by=rank_by,
)
print(f"pairs: {pairs.candidates} candidates in {pairs.rows} rows")

pair_scores = pairs
for model in PAIR_SCORERS:
    kwargs = {k: v for k, v in SCORER_KWARGS.get(model, {}).items() if v is not None}
    print(f"running {model} on {pairs.candidates} pairs ...")
    pair_scores = score_variants(
        complex_,
        pair_scores,
        output_dir=RUN / "pair_scores",
        models=[model],
        max_variants=MAX_PAIRS,
        **kwargs,
    )

pair_frame = pair_scores.frame
pair_columns = list(pair_scores.score_columns)
pair_rank = pair_columns[0] if pair_columns else None
by_candidate = pair_frame.drop_duplicates("variant_id")
if pair_rank:
    by_candidate = by_candidate.sort_values(pair_rank, ascending=False, kind="stable")
    print(f"\npairs ranked on {pair_rank}:")
else:
    print("\nno score column on the pairs; keeping select_pairs' order")
display(by_candidate[["variant_id", *pair_columns]].head(MAX_PAIRS))

## 8. Build the designs

Top singles and scored pairs become one list of designs. Each design's mutant
chain comes from `mutant_sequence`, which applies *every* mutation of the
candidate in one pass - the only honest way to write a pair out.

Two antibody chains are joined with `SUBMIT_LINKER` here, because the veto fold
and the submission are both one antibody chain. Be clear about what that means:
the scoring above ran on a VH/VL pair, and the veto folds the joined scFv. They
are not the same molecule, and the veto is the one that gets the last word.

In [ ]:
designs: list[dict] = []
parent_chains = [sequences[chain] for chain in ab_chains]


def joined(chains: list[str]) -> str | None:
    if len(chains) == 1:
        return chains[0]
    if not SUBMIT_LINKER:
        return None
    return SUBMIT_LINKER.join(chains)


def build(variant_id: str) -> dict:
    mutations = parse_id(variant_id)
    chains = [
        mutant_sequence(complex_.path, mutations, chain)[1] for chain in ab_chains
    ]
    foreign = sorted({c for c, _, _, _ in mutations} - set(ab_chains))
    if foreign:
        raise ValueError(f"{variant_id} mutates non-antibody chain(s) {foreign}")
    return {
        "design": re.sub(r"[^A-Za-z0-9_.-]", "_", variant_id),
        "variant_id": variant_id,
        "mutations": variant_id,
        "n_mutations": len(mutations),
        "positions": tuple(sorted((c, p) for c, p, _, _ in mutations)),
        "chains": chains,
        "sequence": joined(chains),
    }


for row in ranked.head(TOP_SINGLES).itertuples():
    designs.append({**build(str(row.variant_id)), "group": "single"})
for row in by_candidate.itertuples():
    designs.append({**build(str(row.variant_id)), "group": "pair"})

parent = {
    "design": "parent",
    "variant_id": "parent",
    "mutations": "",
    "n_mutations": 0,
    "positions": (),
    "chains": parent_chains,
    "sequence": joined(parent_chains),
    "group": "parent",
}

seen_names = {d["design"] for d in designs}
if len(seen_names) != len(designs):
    raise RuntimeError("two designs sanitised to the same name")
print(f"{len(designs)} design(s) plus the parent")

design_dir = RUN / "designs"
design_dir.mkdir(parents=True, exist_ok=True)
designs_fasta = design_dir / "designs.fasta"
if parent["sequence"] is None:
    print(
        "SUBMIT_LINKER is empty and the antibody has two chains, so there is no\n"
        "single-chain form to fold or submit. The veto cell will skip itself."
    )
else:
    # The parent first: track_a_fold.py reads it as the reference, and the veto
    # compares every design against its seed spread.
    records = [parent, *designs]
    designs_fasta.write_text(
        "".join(f">{d['design']}\n{d['sequence']}\n" for d in records)
    )
    print(f"wrote {designs_fasta} ({len(records)} records)")

design_table = pd.DataFrame(
    [
        {k: v for k, v in d.items() if k not in ("chains", "positions")}
        for d in designs
    ]
)
display(design_table.head(len(designs)))

## 9. The structure veto

`examples/track_a_fold.py` folds every design with the trimmed antigen over
`VETO_SEEDS` seeds and writes one row per (design, seed). It is called as a
subprocess, which is how it is tested, rather than imported for its `main`.

The rule: a design passes if its mean ipTM is at least the parent's mean minus
`VETO_SD_MULTIPLE` times the parent's seed-to-seed SD. The SD is the point of
running several seeds - without it there is no scale on which a drop is large.

These folds are written by the adapter directly rather than through a pipeline
step, so they carry no `DONE` manifest and `run_report` does not see them. The
summary CSV is where they live.

In [ ]:
veto_dir = RUN / "veto"
summary_path = veto_dir / "summary.csv"
veto_stats = pd.DataFrame()

if parent["sequence"] is None:
    print("skipped: no single-chain form to fold (see the previous cell)")
else:
    command = [
        sys.executable,
        str(TRACK_A_FOLD),
        "--scfv", str(designs_fasta),
        "--antigen", str(ANTIGEN),
        "--seeds", str(VETO_SEEDS),
        "--model", PREDICTOR,
        "--out", str(veto_dir),
    ]
    if ANTIGEN_REGION:
        command += ["--antigen-region", ANTIGEN_REGION]
    elif EPITOPE:
        command += ["--epitope", EPITOPE, "--pad", str(PAD)]
    print("$ " + " ".join(command) + "\n")
    completed = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"track_a_fold.py exited {completed.returncode}")

    rows = pd.read_csv(summary_path)
    veto_stats = (
        rows.groupby("complex")
        .agg(
            seeds=("seed", "count"),
            iptm_mean=("iptm", "mean"),
            iptm_sd=("iptm", "std"),
            iptm_min=("iptm", "min"),
            plddt_mean=("plddt", "mean"),
        )
        .reset_index()
        .rename(columns={"complex": "design"})
    )
    display(veto_stats.round(3))

In [ ]:
if veto_stats.empty:
    print("no veto folds, so no veto. Every design is carried forward unvetoed.")
    vetoed = design_table.assign(veto="skipped", iptm_mean=pd.NA, iptm_sd=pd.NA)
    floor = None
else:
    parent_row = veto_stats[veto_stats["design"] == parent["design"]]
    if parent_row.empty:
        raise RuntimeError(f"the veto summary has no {parent['design']!r} row to compare against")
    parent_mean = float(parent_row["iptm_mean"].iloc[0])
    parent_sd = parent_row["iptm_sd"].iloc[0]
    if pd.isna(parent_sd):
        raise RuntimeError(
            "the parent folded on one seed, so there is no seed SD and no scale for "
            "the veto. Raise VETO_SEEDS to at least 2."
        )
    parent_sd = float(parent_sd)
    floor = parent_mean - VETO_SD_MULTIPLE * parent_sd
    print(
        f"parent ipTM {parent_mean:.3f} +/- {parent_sd:.3f} over "
        f"{int(parent_row['seeds'].iloc[0])} seed(s)"
    )
    print(f"veto floor: {floor:.3f} (mean - {VETO_SD_MULTIPLE} x SD)\n")

    vetoed = design_table.merge(veto_stats, on="design", how="left")
    missing = vetoed[vetoed["iptm_mean"].isna()]["design"].tolist()
    if missing:
        print(f"warning: no fold for {len(missing)} design(s): {', '.join(missing)}")
    vetoed["veto"] = [
        "no fold" if pd.isna(value) else ("pass" if value >= floor else "fail")
        for value in vetoed["iptm_mean"]
    ]
    print(vetoed["veto"].value_counts().to_string())
    display(
        vetoed.sort_values("iptm_mean", ascending=False)[
            ["design", "group", "n_mutations", "iptm_mean", "iptm_sd", "plddt_mean", "veto"]
        ].round(3)
    )

## 10. Pick K and write the submission

Order: the veto survivors by mean ipTM where there is one, otherwise by the
ranking column. Then walk the list and skip a design that shares more than
`MAX_SHARED_POSITIONS` mutated positions with one already taken, so K designs
are not K variations on the same site - organisers have rewarded difference, and
ten near-copies fail together.

If fewer than K survive, this does not pad the list. Relaxing a gate is a
decision with a reason, taken in the parameters cell and re-run.

In [ ]:
pool = vetoed[vetoed["veto"] != "fail"].copy()
position_lookup = {d["design"]: set(d["positions"]) for d in designs}

if "iptm_mean" in pool and pool["iptm_mean"].notna().any():
    pool = pool.sort_values("iptm_mean", ascending=False, kind="stable")
    order = "veto ipTM"
else:
    scores = {str(r.variant_id): getattr(r, rank_by) for r in ranked.itertuples()}
    pool["fallback"] = [scores.get(v, float("-inf")) for v in pool["variant_id"]]
    pool = pool.sort_values("fallback", ascending=False, kind="stable")
    order = f"{rank_by} (no veto)"
print(f"ordering by {order}")

kept: list[str] = []
taken: set[tuple[str, int]] = set()
for row in pool.itertuples():
    if len(kept) >= K:
        break
    positions_here = position_lookup[row.design]
    shared = positions_here & taken
    if len(shared) > MAX_SHARED_POSITIONS:
        print(f"skipped {row.design}: shares {len(shared)} position(s) with a kept design")
        continue
    kept.append(row.design)
    taken |= positions_here

final = pool[pool["design"].isin(kept)].copy()
final["rank"] = [kept.index(name) + 1 for name in final["design"]]
final = final.sort_values("rank")
if len(final) < K:
    print(
        f"\nonly {len(final)} design(s) of the requested {K} survived. Not padded: "
        "relax a gate in the parameters cell, with a reason, and re-run."
    )

out = RUN / "submission"
out.mkdir(parents=True, exist_ok=True)
by_design = {d["design"]: d for d in designs}
final["sequence"] = [by_design[name]["sequence"] for name in final["design"]]
final.to_csv(out / "candidates.csv", index=False)
(out / "candidates.fasta").write_text(
    "".join(
        f">{row.design} rank={row.rank} mutations={row.mutations}\n{row.sequence}\n"
        for row in final.itertuples()
    )
)
print(f"\nwrote {out / 'candidates.csv'} and {out / 'candidates.fasta'}")
display(
    final[["rank", "design", "group", "mutations", "n_mutations", "iptm_mean", "veto"]]
)

## 11. The run report

`run_report` reads the `DONE` manifests `design_pipeline` writes and scores them
against `thresholds.toml`. It covers the parent fold; the veto folds have no
manifests, as the veto cell said, so the submission table above is the record for
those.

`ipsae` is in `thresholds.toml` as the headline metric and **nothing in this
repository computes it** - `docs/filtering_strategy.md` lists it as still to add.
If the organisers rank on ipSAE, run their script on `candidates.fasta` and treat
the ipTM here as a proxy with no calibration behind it.

In [ ]:
report_dir = RUN / "report"
command = [
    sys.executable, "-m", "run_report", "build", str(RUN),
    "--run-id", RUN.name,
    "--out", str(report_dir),
    "--parent", "parent",
    "--code-dir", str(REPO),
]
print("$ " + " ".join(command) + "\n")
completed = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
print(completed.stdout or "(no stdout)")
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
if completed.returncode == 0:
    print(f"\nreport: {report_dir / 'report.md'}")
else:
    print(
        f"\nrun_report exited {completed.returncode}. It needs DONE manifests with "
        "metrics from thresholds.toml; the submission files above do not depend on it."
    )